# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

Source Used: Google Gemini - Used Google Gemini to help me understand syntax but also used it to explain to me why I was getting certain errors

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
import pandas as pd
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]
orders_df = pd.json_normalize(orders_json, record_path=['lines'], meta=['order', 'vendor_id'])
vendors_df = pd.json_normalize(vendors_json)
df = orders_df.merge(vendors_df, on="vendor_id", how='left', indicator='True')

**One row of my result is one:** One row of my result is one row of item, qty, vendor_id, name, and zone

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
lines = pd.json_normalize(orders_json, record_path=['lines'], meta=['order', 'vendor_id'])
assert len(lines) == 4, "Expect row output of 4"
assert lines['qty'].sum() == 7, "Quantities should sum to 7"

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

The vendor that did not match was V-07 and only one unit was attached which was item: Hot Dog, qty: 3, order: 3, vendor_id: V-07, name: NaN, and zone: NaN

In [3]:
# TODO
orders_df = pd.json_normalize(orders_json, record_path=['lines'], meta=['order', 'vendor_id'])
vendors_df = pd.json_normalize(vendors_json)
joined = orders_df.merge(vendors_df, on="vendor_id", how='left', indicator=True)
joined


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# TODO
# The unmatched vendor appears in the table as NaN as the vendor sold some items but we do not have the vendor assigned in the directory
print(df.groupby('name', dropna=False)['qty'].sum())

name
Cav Merch       1
Hoos Burgers    3
NaN             3
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
# Zone A sold the most items, my total is 7 as I did not drop the NaN zone however if I had dropped the NaN my total would be 4 as the other 3 item was sold by the NaN zone
print(df.groupby('zone', dropna=False)['qty'].sum())

zone
A      4
NaN    3
Name: qty, dtype: int64


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# Utilize a for loop to add a empty list for lines in order 5 to flatten json
count = 0
for value in ragged_json:
    if value.get('lines') == None:
        ragged_json[count]['lines'] = []
    count += 1

# TODO
cleaned_df = pd.json_normalize(ragged_json, record_path=['lines'], meta=['order', 'vendor_id'])
cleaned_df

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is the useful shape here as it allows for cleaner processing of what items the vendor sold for example V-01 sold both cheeseburger and soda however if we wanted to answer a question of how many qty of soda did V-01 sell by having it seperate it makes it easier for us to find the qty and then output the result compared to if we did not do that way it would become signficantly harder as we would have to take that list of directory and iterate through each of the items untill we found what we are looking for.

b) The practical difference between a missing quantity and a quantity of zero is if the value is NaN or not. The reason why this breaks downstream is if we do not differentiate the difference between them we might later get one missing values that is NaN or get zero values that is also NaN causing us to incorrectly use missing values or incorrectly throw away valid values.